# BEE 4750 Lab 4: Scenario Trees and Sequential Decisions

**Name**:

**ID**:

> **Due Date**
>
> Tuesday, November 24, 9:00pm

## Setup

In [1]:
import Pkg
Pkg.activate(".")
Pkg.instantiate()

## Introduction

### The Reservoir

A water-supply reservoir serves a city and an irrigation district
through a four-month season, June to September. Storage and flows are in
units of one million m<sup>3</sup>.

- The reservoir holds up to 8 units and is full on June 1.
- At the start of each month the operator chooses a **release** $r$, in
  whole units, before knowing that month’s inflow. The release can be no
  more than what is stored and no more than the demand of 4 units.
- Then the month’s **inflow** $I$ arrives, and anything above 8 units
  spills over, so with $s$ units stored at the start of the month, next
  month’s storage is $\min(s - r + I,\ 8)$.

The first unit released in a month covers the city’s essential use.
Later units cover the rest of the city’s demand and then irrigation, so
each additional unit released has diminishing value:

| Release in a month (units) |   0 |   1 |   2 |   3 |   4 |
|:---------------------------|----:|----:|----:|----:|----:|
| **Benefit** (\$100,000)    |   0 |  10 |  17 |  20 |  21 |

Each month is either **dry** or **wet**. Late-summer storms are smaller,
so a wet month brings less water in August and September:

| Month                         | June | July | August | September |
|:------------------------------|-----:|-----:|-------:|----------:|
| Inflow in a dry month (units) |    0 |    0 |      0 |         0 |
| Inflow in a wet month (units) |    4 |    4 |      2 |         2 |

With no forecasted information, each month is dry with probability 0.5,
independent of the other months. A seasonal climate forecast shifts
those operational probabilities.

Water left in the reservoir on October 1 carries over to next year. By
default it is worth nothing (the reservoir refills over the winter).

A common default rule, the **standard operating policy**, releases the
full demand whenever the reservoir holds it: $r = \min(s, 4)$.

The code below stores these values and defines two functions that the
rest of the lab uses: `benefit(r)` looks up the benefit of releasing `r`
units in a month, and `next_storage(s, r, inflow)` gives next month’s
storage from this month’s storage `s`, release `r`, and inflow.

In [1]:
# Storage and flows are in units of 1 million cubic meters, and
# benefits in $100,000
capacity = 8                          # storage capacity
start_storage = 8                     # storage on June 1 (full)
max_release = 4                       # demand: most usable in a month
benefit_table = [0, 10, 17, 20, 21]   # benefit of releasing 0 to 4
months = ["June", "July", "August", "September"]
n_months = length(months)
inflow_dry = [0, 0, 0, 0]             # dry-month inflow, June to Sept.
inflow_wet = [4, 4, 2, 2]             # wet-month inflow, June to Sept.

# Benefit of releasing r units in a month (releasing 0 is the first
# entry of the table)
benefit(r) = benefit_table[r + 1]

# Storage at the start of next month: release r from storage s, then
# the month's inflow arrives; anything above capacity spills
next_storage(s, r, inflow) = min(s - r + inflow, capacity)

> **Two Ways to Work Through This Lab**
>
> You can do the coding either in this notebook (`lab04.ipynb`) or in
> the plain script `lab04.jl`, whichever you prefer; both contain the
> same problems. Either way, the written parts go on the separate
> **Activity Sheet** (handed out in class), which you can hand in at the
> end of class or submit to Gradescope as a PDF by 9pm on Tuesday; you
> don’t need to submit your code.

## Problem 1: The Scenario Tree

Before writing any code, complete Problem 1 on the Activity Sheet: write
down the transition function, and sketch the scenario tree for June and
July.

## Problem 2: Dynamic Programming

Start with Problem 2a on the Activity Sheet: write down the Bellman
equation for this reservoir. Then implement it.

The function below solves the problem by dynamic programming, working
backward from September (backward induction). Its inputs are `prob_dry`,
the probability that a month is dry, and `carryover_value`, the value
per unit of water left on October 1. `value_table` holds $V_t(s)$, and
`policy_table` holds the release that achieves it. Storage $s$ is kept
in row $s + 1$ of each table, because Julia arrays start at 1. Month $t$
(1 for June to 4 for September) is in column $t$, and column 5 of
`value_table` holds the terminal value $V_5(s)$.

Each of the four TODOs is one part of your Bellman equation: the
terminal value, the largest release allowed, next month’s storage after
a dry and after a wet month, and this month’s benefit plus the expected
`value_table` entry at next month’s storage. Replace each `missing` with
the code for that part.

In [0]:
function backward_induction(prob_dry, carryover_value)
    # best expected total from month t on, and the release that gets it
    value_table = zeros(capacity + 1, n_months + 1)
    policy_table = zeros(Int, capacity + 1, n_months)
    for s in 0:capacity
        # TODO: replace `missing` with the value of s units of water
        # left on October 1
        value_table[s + 1, n_months + 1] = missing
    end
    for t in n_months:-1:1
        for s in 0:capacity
            best_total = -Inf
            # TODO: replace `missing` with the largest release allowed
            # from storage s
            for r in 0:missing
                # TODO: replace each `missing` with next month's storage
                # after a dry month and after a wet month
                storage_if_dry = missing
                storage_if_wet = missing
                # TODO: replace `missing` with this month's benefit plus
                # the expected value_table entry at next month's storage
                total = missing
                # keep the best release (ties keep the smaller one)
                if total > best_total
                    best_total = total
                    value_table[s + 1, t] = total
                    policy_table[s + 1, t] = r
                end
            end
        end
    end
    return value_table, policy_table
end

Run it assuming equal probabilities of wet and dry months
(`prob_dry = 0.5`) and no value on water left on October 1
(`carryover_value = 0`). The code below prints the expected total
benefit and June’s release, starting from a full reservoir:

In [0]:
value_table, policy_table = backward_induction(0.5, 0)
println("Expected total benefit: ", value_table[start_storage + 1, 1])
println("June release: ", policy_table[start_storage + 1, 1])

Print the whole policy, the best release for each storage level and
month:

In [0]:
println("Storage: releases in June, July, August, September")
for s in 0:capacity
    println(s, ": ", policy_table[s + 1, :])
end

The function below follows a policy through one sequence of inflows and
returns the releases and the storage left on October 1. Complete its two
TODOs: the release the policy gives for this month and storage, and the
storage at the start of next month. Running the cell then prints the
releases in the driest summer (every month dry) and the wettest summer
(every month wet). Record these, the expected total benefit, and June’s
release in Table 1 on the Activity Sheet.

In [0]:
function follow_policy(policy_table, inflows)
    releases = zeros(Int, n_months)
    storage = start_storage
    for t in 1:n_months
        # TODO: replace `missing` with the policy's release in month t
        # from this storage
        releases[t] = missing
        # TODO: replace `missing` with the storage at the start of
        # next month
        storage = missing
    end
    return releases, storage
end

driest_releases, driest_storage_left =
    follow_policy(policy_table, inflow_dry)
wettest_releases, wettest_storage_left =
    follow_policy(policy_table, inflow_wet)
println("Driest summer: ", driest_releases)
println("Wettest summer: ", wettest_releases)

## Problem 3 (Optional): Extensions of the Basic DP Model

If you finish Problems 1 and 2, start with Problem 3a on the Activity
Sheet, which compares the policy from Problem 2 with the standard
operating policy. Work out the standard operating policy’s releases in
the driest and wettest summers by hand, and each summer’s total benefit
under both policies.

For Problem 3b, re-solve the problem with `backward_induction` and
`follow_policy` for each case in Table 2 on the Activity Sheet: a wet
outlook, a drought outlook, and values of 4 and 8 per unit for water
left on October 1. For each case, find June’s release, the expected
total value (the expected benefit plus the expected value of the water
left on October 1), and the releases in the driest and wettest summers.

In [0]:
outlook_prob_dry = [0.2, 0.8, 0.5, 0.5]
carryover_values = [0, 0, 4, 8]
for i in 1:length(outlook_prob_dry)
    # TODO: solve the case with outlook_prob_dry[i] and
    # carryover_values[i], then print June's release, the expected
    # total value, and the releases in the driest and wettest summers
end

Record your results in Table 2, then answer Problem 3c on the Activity
Sheet.